# Migration des Données Médicales vers MongoDB

## Objectif
Ce notebook a pour but de démontrer les étapes nécessaires pour migrer un fichier de données médicales au format CSV vers une base de données MongoDB, avec les étapes suivantes :
1. Chargement et validation des données.
2. Nettoyage et export des données.
3. Insertion des données dans MongoDB.
4. Vérification des documents insérés dans la base de données.

## Fichiers requis
- **healthcare_dataset.csv** : Fichier source contenant les données médicales. Ce fichier doit être placé dans le répertoire `data/raw/`.

## Résultat attendu
Les données nettoyées seront enregistrées dans `data/processed/healthcare_data_cleaned.csv` et insérées dans une base MongoDB locale.


In [24]:
# Importation des bibliothèques nécessaires
import pandas as pd  # Pour la manipulation des données
import os  # Pour interagir avec le système de fichiers
from loguru import logger  # Pour gérer les logs
from pymongo import MongoClient  # Pour se connecter à MongoDB
from pathlib import Path  # Pour gérer les chemins de fichiers et répertoires
import sys  # Pour modifier le chemin d'import des modules
import kagglehub  # Pour télécharger les datasets Kaggle
from skimpy import skim  # Résumés statistiques enrichis pour les DataFrames
import warnings   # Suppression des avertissements inutiles

# --- Configuration des Affichages pandas ---
pd.set_option('display.max_columns', None)  # Affiche toutes les colonnes d'un DataFrame
pd.set_option('display.max_rows', 100)      # Affiche jusqu'à 100 lignes dans un DataFrame
pd.set_option('display.width', 400)         # Évite les retours à la ligne inutiles

# --- Suppression des Avertissements Inutiles ---
warnings.filterwarnings("ignore")

# Configuration des logs
# Les logs sont enregistrés dans un fichier avec une rotation automatique pour limiter la taille
logger.add("../logs/migration.log", rotation="500 MB")
logger.info("Démarrage du notebook")

# Définir le répertoire de base pour un notebook ou un script
# Si on est dans un notebook, on remonte au répertoire parent
NOTEBOOK_PATH = Path(os.getcwd())
if NOTEBOOK_PATH.name == "notebooks":
    BASE_DIR = NOTEBOOK_PATH.parent
else:
    BASE_DIR = NOTEBOOK_PATH

# Ajouter le chemin du répertoire 'src' au sys.path
sys.path.append(str(BASE_DIR / "src"))

# Définir les chemins des répertoires à partir du répertoire de base
RAW_DIR = BASE_DIR / "data/raw"  # Répertoire pour les fichiers bruts
PROCESSED_DIR = BASE_DIR / "data/processed"  # Répertoire pour les fichiers nettoyés

# --- Fonction pour configurer et loguer les répertoires ---
def setup_and_log_directories(base_dir, raw_dir, processed_dir):
    """
    Configure les répertoires pour les fichiers bruts et nettoyés.
    Crée les répertoires si nécessaire et logue leur statut en chemins relatifs avec des descriptions.
    """
    try:
        # Convertir les chemins en chemins relatifs
        base_dir_relative = base_dir.name  # Nom du répertoire racine
        raw_dir_relative = raw_dir.relative_to(base_dir)
        processed_dir_relative = processed_dir.relative_to(base_dir)

        # Log du répertoire racine
        logger.info(f"Répertoire racine pour le projet défini : {base_dir_relative}")

        # Crée ou vérifie le répertoire des fichiers bruts
        raw_dir.mkdir(parents=True, exist_ok=True)
        if raw_dir.exists():
            logger.info(f"Répertoire pour les fichiers bruts (RAW_DIR) configuré : {base_dir_relative}\\{raw_dir_relative}")
        else:
            logger.error(f"Erreur lors de la configuration du répertoire pour les fichiers bruts : {raw_dir_relative}")

        # Crée ou vérifie le répertoire des fichiers nettoyés
        processed_dir.mkdir(parents=True, exist_ok=True)
        if processed_dir.exists():
            logger.info(f"Répertoire pour les fichiers nettoyés (PROCESSED_DIR) configuré : {base_dir_relative}\\{processed_dir_relative}")
        else:
            logger.error(f"Erreur lors de la configuration du répertoire pour les fichiers nettoyés : {processed_dir_relative}")

    except Exception as e:
        # Log des erreurs en cas de problème
        logger.error(f"Erreur lors de la configuration des répertoires : {e}")
        raise

# 📂 Appel de la fonction avec vos variables
setup_and_log_directories(BASE_DIR, RAW_DIR, PROCESSED_DIR)


logger.info("Répertoires vérifiés et créés avec succès.")

2025-01-02 10:40:12.866 | INFO     | __main__:<module>:23 - Démarrage du notebook
2025-01-02 10:40:12.868 | INFO     | __main__:setup_and_log_directories:53 - Répertoire racine pour le projet défini : migration-donnees-mongodb
2025-01-02 10:40:12.869 | INFO     | __main__:setup_and_log_directories:58 - Répertoire pour les fichiers bruts (RAW_DIR) configuré : migration-donnees-mongodb\data\raw
2025-01-02 10:40:12.871 | INFO     | __main__:setup_and_log_directories:65 - Répertoire pour les fichiers nettoyés (PROCESSED_DIR) configuré : migration-donnees-mongodb\data\processed
2025-01-02 10:40:12.872 | INFO     | __main__:<module>:78 - Répertoires vérifiés et créés avec succès.


In [25]:
# -------------------------------------------
# Téléchargement et Chargement des Données
# -------------------------------------------

# Téléchargement du dataset depuis Kaggle
try:
    # Télécharge le dataset en utilisant KaggleHub
    dataset_path = kagglehub.dataset_download("prasad22/healthcare-dataset")
    logger.info(f"Dataset téléchargé avec succès")
except Exception as e:
    # Gère les erreurs de téléchargement et les logue
    logger.error(f"Erreur lors du téléchargement du dataset : {e}")
    raise  # Arrête l'exécution en cas d'échec

# Étape 2 : Lister les fichiers disponibles dans le dossier téléchargé
try:
    # Liste tous les fichiers dans le répertoire du dataset
    files = os.listdir(dataset_path)
    logger.info(f"Fichiers disponibles dans le dataset : {files}")
except Exception as e:
    # Log en cas d'échec de la lecture du répertoire
    logger.error(f"Erreur lors de la lecture des fichiers dans le répertoire {dataset_path} : {e}")
    raise

# Étape 3 : Chargement d'un fichier spécifique
# Nom du fichier attendu
dataset_file = Path(dataset_path) / "healthcare_dataset.csv"
try:
    # Charge le fichier CSV en mémoire en tant que DataFrame pandas
    data = pd.read_csv(dataset_file)
    logger.info(f"Fichier '{dataset_file.name}' chargé avec succès. "
                f"Nombre de lignes : {data.shape[0]}. Nombre de colonnes : {data.shape[1]}.")
except FileNotFoundError:
    # Log en cas de fichier manquant
    logger.error(f"Le fichier spécifié '{dataset_file.name}' est introuvable dans le dataset.")
    raise
except Exception as e:
    # Log pour tout autre type d'erreur
    logger.error(f"Erreur lors du chargement du fichier '{dataset_file.name}' : {e}")
    raise


2025-01-02 10:40:13.339 | INFO     | __main__:<module>:9 - Dataset téléchargé avec succès
2025-01-02 10:40:13.342 | INFO     | __main__:<module>:19 - Fichiers disponibles dans le dataset : ['healthcare_dataset.csv']
2025-01-02 10:40:13.449 | INFO     | __main__:<module>:31 - Fichier 'healthcare_dataset.csv' chargé avec succès. Nombre de lignes : 55500. Nombre de colonnes : 15.


In [26]:
# Aperçu des données
# Afficher les 5 premières lignes pour un aperçu rapide du contenu
logger.info(f"Aperçu des données : \n{data.head()}\n")

# Résumé statistique avec skim
# Fournit une vue d'ensemble rapide des colonnes, types, valeurs manquantes, etc.
logger.info(f"Résumé pour le dataset 'healthcare_dataset':")
skim(data)  # Assurez-vous que skimpy est installé : pip install skimpy

# Séparateur visuel dans les logs
logger.info(f"{'=' * 60}")

# Types des colonnes
# Affiche les types des colonnes pour identifier d'éventuels problèmes
logger.info(f"📊 Types des colonnes pour le DataFrame 'healthcare'")
logger.info(f"{'-' * 60}")

# Boucle pour afficher les détails de chaque colonne
# Cela inclut le type de données pandas et les types Python uniques rencontrés
for col in data.columns:
    col_type = data[col].dtype  # Type de colonne selon pandas (int64, object, etc.)
    unique_types = data[col].apply(lambda x: type(x).__name__).unique()  # Types Python rencontrés
    logger.info(f"🔹 {col} : {col_type} | Valeurs : {', '.join(unique_types)}")

# Séparateur final
logger.info(f"{'=' * 60}\n")


2025-01-02 10:40:13.463 | INFO     | __main__:<module>:3 - Aperçu des données : 
            Name  Age  Gender Blood Type Medical Condition Date of Admission            Doctor                    Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31     Matthew Smith             Sons and Miller         Blue Cross    18856.281306          328         Urgent     2024-02-02  Paracetamol        Normal
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   Samantha Davies                     Kim Inc           Medicare    33643.327287          265      Emergency     2019-08-26    Ibuprofen  Inconclusive
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22  Tiffany Mitchell                    Cook PLC              Aetna    27955.096079          205      Emergency     2022-10-07      Aspirin        Normal
3   and

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 55500  │ │ string      │ 12    │                                                          │
│ │ Number of columns │ 15     │ │ int32       │ 2     │                                                          │
│ └───────────────────┴────────┘ │ float64     │ 1     │                                                          │
│                                └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column_name       ┃ NA  ┃ NA %   ┃ mean    ┃ sd     ┃ p0     ┃ p25    ┃ p50    ┃ p75    ┃ p100   ┃ hist    ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩  │
│ │ Age               │   0 │      0 │   51.54 │   19.6 │     13 │     35 │     52 │     68 │     89 │ ▅▇▇▇▇▅  │  │
│ │ Billing Amount    │   0 │      0 │   25540 │  14210 │  -2008 │  13240 │  25540 │  37820 │  52760 │ ▅▇▇▇▇▆  │  │
│ │ Room Number       │   0 │      0 │   301.1 │  115.2 │    101 │    202 │    302 │    401 │    500 │ ▇▇▇▇▇▇  │  │
│ └───────────────────┴─────┴────────┴─────────┴────────┴────────┴────────┴────────┴────────┴────────┴─────────┘  │
│                                                     string                                                      │
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓  │
│ ┃ column_name                        ┃ NA     ┃ NA %      ┃ words per row             ┃ total words          ┃  │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩  │
│ │ Name                               │      0 │         0 │                         2 │               113390 │  │
│ │ Gender                             │      0 │         0 │                         1 │                55500 │  │
│ │ Blood Type                         │      0 │         0 │                         1 │                55500 │  │
│ │ Medical Condition                  │      0 │         0 │                         1 │                55500 │  │
│ │ Date of Admission                  │      0 │         0 │                         1 │                55500 │  │
│ │ Doctor                             │      0 │         0 │                         2 │               113516 │  │
│ │ Hospital                           │      0 │         0 │                       2.4 │               133009 │  │
│ │ Insurance Provider                 │      0 │         0 │                       1.2 │                66559 │  │
│ │ Admission Type                     │      0 │         0 │                         1 │                55500 │  │
│ │ Discharge Date                     │      0 │         0 │                         1 │                55500 │  │
│ │ Medication                         │      0 │         0 │                         1 │                55500 │  │
│ │ Test Results                       │      0 │         0 │                         1 │                55500 │  │
│ └────────────────────────────────────┴────────┴───────────┴───────────────────────────┴──────────────────────┘  │
╰────────────────────────────────────────────────────── 

2025-01-02 10:40:13.791 | INFO     | __main__:<module>:11 - ============================================================
2025-01-02 10:40:13.793 | INFO     | __main__:<module>:15 - 📊 Types des colonnes pour le DataFrame 'healthcare'
2025-01-02 10:40:13.794 | INFO     | __main__:<module>:16 - ------------------------------------------------------------
2025-01-02 10:40:13.805 | INFO     | __main__:<module>:23 - 🔹 Name : object | Valeurs : str
2025-01-02 10:40:13.815 | INFO     | __main__:<module>:23 - 🔹 Age : int64 | Valeurs : int
2025-01-02 10:40:13.825 | INFO     | __main__:<module>:23 - 🔹 Gender : object | Valeurs : str
2025-01-02 10:40:13.835 | INFO     | __main__:<module>:23 - 🔹 Blood Type : object | Valeurs : str
2025-01-02 10:40:13.845 | INFO     | __main__:<module>:23 - 🔹 Medical Condition : object | Valeurs : str
2025-01-02 10:40:13.854 | INFO     | __main__:<module>:23 - 🔹 Date of Admission : object | Valeurs : str
2025-01-02 10:40:13.864 | INFO     | __main__:<module>:23 - 🔹 

In [27]:
# Identification des doublons
data['is_duplicate'] = data.duplicated(keep=False)

# Filtrer les lignes marquées comme doublons
duplicates = data[data['is_duplicate']].copy()

# Ajouter une colonne 'is_primary' pour identifier la première occurrence
duplicates['is_primary'] = ~duplicates.duplicated(keep='first')

# Normaliser les colonnes avant le traitement
columns_to_normalize = [col for col in data.columns if col not in ['is_duplicate', 'is_primary']]
for col in columns_to_normalize:
    if data[col].dtype == 'object':  # Normaliser uniquement les colonnes de type texte
        duplicates[col] = duplicates[col].str.strip().str.lower()  # Supprimer les espaces et uniformiser les majuscules

# Ajouter un identifiant temporaire unique pour chaque ligne
duplicates = duplicates.reset_index(drop=False).rename(columns={'index': 'temp_id'})

# Séparer les lignes principales et les doublons
primary_rows = duplicates[duplicates['is_primary']].copy()
duplicate_rows = duplicates[~duplicates['is_primary']].copy()

# Ajouter un identifiant unique pour les groupes de doublons
primary_rows['group_id'] = range(1, len(primary_rows) + 1)

# Associer les doublons à leurs lignes principales
duplicates_with_group = duplicate_rows.merge(
    primary_rows[columns_to_normalize + ['group_id']], 
    how='left', 
    on=columns_to_normalize, 
    suffixes=('', '_primary')
)

# Vérification des colonnes générées après le merge
generated_columns = duplicates_with_group.columns
logger.info(f"Colonnes générées après le merge : {generated_columns}")

# Comparer chaque colonne entre les lignes principales et leurs doublons
diff_columns = []  # Initialisation de la liste des colonnes de différences
for col in columns_to_normalize:
    primary_col = f"{col}_primary"
    if primary_col in duplicates_with_group.columns:
        diff_col = f"{col}_diff"
        duplicates_with_group[diff_col] = duplicates_with_group[col] != duplicates_with_group[primary_col]
        diff_columns.append(diff_col)  # Ajouter à la liste des colonnes de différences
    else:
        logger.warning(f"La colonne '{primary_col}' n'existe pas. Comparaison ignorée.")

# Ajouter la liste des groupes concernés par des doublons
if not duplicates_with_group.empty:  # Vérifie si le DataFrame n'est pas vide
    group_ids = duplicates_with_group['group_id'].unique()  # Extraire les identifiants des groupes
    logger.info(f"Groupes concernés par des doublons : {group_ids.tolist()}")
else:
    logger.info("Aucun groupe concerné par des doublons.")


# Résultat final pour analyse
logger.info(f"Groupes de doublons avec colonnes de différences : \n{duplicates_with_group.head()}")
logger.info(f"Colonnes avec des différences détectées : {diff_columns}")


2025-01-02 10:40:14.010 | INFO     | __main__:<module>:36 - Colonnes générées après le merge : Index(['temp_id', 'Name', 'Age', 'Gender', 'Blood Type', 'Medical Condition', 'Date of Admission', 'Doctor', 'Hospital', 'Insurance Provider', 'Billing Amount', 'Room Number', 'Admission Type', 'Discharge Date', 'Medication', 'Test Results', 'is_duplicate', 'is_primary', 'group_id'], dtype='object')
2025-01-02 10:40:14.012 | WARNING  | __main__:<module>:47 - La colonne 'Name_primary' n'existe pas. Comparaison ignorée.
2025-01-02 10:40:14.013 | WARNING  | __main__:<module>:47 - La colonne 'Age_primary' n'existe pas. Comparaison ignorée.
2025-01-02 10:40:14.014 | WARNING  | __main__:<module>:47 - La colonne 'Gender_primary' n'existe pas. Comparaison ignorée.
2025-01-02 10:40:14.015 | WARNING  | __main__:<module>:47 - La colonne 'Blood Type_primary' n'existe pas. Comparaison ignorée.
2025-01-02 10:40:14.015 | WARNING  | __main__:<module>:47 - La colonne 'Medical Condition_primary' n'existe pas. 

In [28]:
# Liste des groupes à analyser
groups_to_check = [81, 21, 279, 61, 211, 383, 112, 196, 231, 55, 309, 508, 172, 384, 387, 202, 450, 92, 481, 432, 223, 307]  # Remplacez par les identifiants des groupes que vous souhaitez analyser

# Filtrer les lignes principales et les doublons pour les groupes sélectionnés
groups_primary = primary_rows[primary_rows['group_id'].isin(groups_to_check)]
groups_duplicates = duplicates_with_group[duplicates_with_group['group_id'].isin(groups_to_check)]

# Combiner les lignes principales et les doublons pour analyse
groups_to_inspect = pd.concat([groups_primary, groups_duplicates])

# Trier les résultats par 'group_id' pour regrouper les lignes par groupe
groups_to_inspect = groups_to_inspect.sort_values(by=['group_id', 'is_primary'], ascending=[True, False])

# Afficher le contenu des groupes pour validation
logger.info(f"Contenu complet des groupes triés par 'group_id' : \n{groups_to_inspect}")


2025-01-02 10:40:14.043 | INFO     | __main__:<module>:15 - Contenu complet des groupes triés par 'group_id' : 
     temp_id                 Name  Age  Gender Blood Type Medical Condition Date of Admission                 Doctor                      Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results  is_duplicate  is_primary  group_id
20      1336     kimberly vasquez   26    male         a-           obesity        2023-10-23       jennifer bennett                     cowan inc   unitedhealthcare    38142.109678          313         urgent     2023-11-18   penicillin      abnormal          True        True        21
1      50040     kimberly vasquez   26    male         a-           obesity        2023-10-23       jennifer bennett                     cowan inc   unitedhealthcare    38142.109678          313         urgent     2023-11-18   penicillin      abnormal          True       False        21
54      3960      regina

In [29]:
# Identification des doublons
data['is_duplicate'] = data.duplicated(keep=False)

# Filtrer les lignes marquées comme doublons
duplicates = data[data['is_duplicate']].copy()

# Ajouter une colonne 'is_primary' pour identifier la première occurrence
duplicates['is_primary'] = ~duplicates.duplicated(keep='first')

# Rattacher la colonne 'is_primary' à l'ensemble du DataFrame
data['is_primary'] = data.index.isin(duplicates.index[duplicates['is_primary']])

# Garder uniquement les lignes principales et les lignes sans doublons
cleaned_data = data[data['is_primary'] | ~data['is_duplicate']].copy()

# Supprimer les colonnes ajoutées pour l’analyse des doublons
columns_to_remove = ['is_primary', 'is_duplicate', 'group_id']
cleaned_data.drop(columns=columns_to_remove, inplace=True, errors='ignore')

# Afficher le résultat final
logger.info(f"Nombre de lignes après suppression des doublons : {cleaned_data.shape[0]}")
logger.info(f"Aperçu des données nettoyées : \n{cleaned_data.head()}")


2025-01-02 10:40:14.116 | INFO     | __main__:<module>:21 - Nombre de lignes après suppression des doublons : 54966
2025-01-02 10:40:14.120 | INFO     | __main__:<module>:22 - Aperçu des données nettoyées : 
            Name  Age  Gender Blood Type Medical Condition Date of Admission            Doctor                    Hospital Insurance Provider  Billing Amount  Room Number Admission Type Discharge Date   Medication  Test Results
0  Bobby JacksOn   30    Male         B-            Cancer        2024-01-31     Matthew Smith             Sons and Miller         Blue Cross    18856.281306          328         Urgent     2024-02-02  Paracetamol        Normal
1   LesLie TErRy   62    Male         A+           Obesity        2019-08-20   Samantha Davies                     Kim Inc           Medicare    33643.327287          265      Emergency     2019-08-26    Ibuprofen  Inconclusive
2    DaNnY sMitH   76  Female         A-           Obesity        2022-09-22  Tiffany Mitchell              